# DataCops — Key Vault + ADLS 연결 설정 가이드
 
> Databricks에서 Key Vault 시크릿 읽기 + ADLS 연결
 
---
 
## 문제 요약
 
Databricks 노트북에서 Key Vault 시크릿을 읽으려 할 때 아래 오류 발생.
 
```
(Forbidden) Caller is not authorized to perform action on resource.
Caller: appid=0089b876-e1dc-4718-a834-e91f55a464d3
Action: 'Microsoft.KeyVault/vaults/secrets/getSecret/action'
Assignment: (not found)
```
 
**원인:** Databricks 내부 앱(`0089b876`)이 Key Vault 읽기 권한 없음
**해결:** Key Vault IAM에서 Databricks 앱에 `Key Vault 비밀 사용자` 역할 직접 부여
 
---
 
## Azure 리소스 정보
 
| 항목 | 값 |
|------|-----|
| Key Vault 이름 | `kv-sense-team4` |
| Key Vault URL | `https://kv-sense-team4.vault.azure.net/` |
| ADLS 계정 | `datacopsadls` |
| 리소스 그룹 | `3dt-final-team4` |
| 구독 ID | `27db5ec6-d206-4028-b5e1-6004dca5eeef` |
| 테넌트 ID | `5fb256f0-fbf2-40d2-81d5-bac1b32c419d` |
| Service Principal | `datacops-sp` |
| SP 클라이언트 ID | `e17113f6-050c-40e3-80ea-a2f4d0a8976b` |
| Databricks URL | `https://adb-7405616801239507.7.azuredatabricks.net` |
 
---
 
## Key Vault에 등록해야 할 시크릿 목록
 
| 시크릿 이름 | 값 출처 |
|------------|---------|
| `adls-client-id` | datacops-sp 애플리케이션(클라이언트) ID |
| `adls-client-secret` | datacops-sp → 인증서 및 비밀 → 값 |
| `adls-tenant-id` | `5fb256f0-fbf2-40d2-81d5-bac1b32c419d` |
| `azure-openai-key` | datacops-openai → 키 및 엔드포인트 → 키1 |
| `azure-openai-endpoint` | datacops-openai → 키 및 엔드포인트 → 엔드포인트 |
 
---
 
## 해결 방법 — Key Vault IAM 권한 부여
 
### Step 1 — datacops-sp에 ADLS 권한 부여
 
```
Azure Portal
→ datacopsadls 스토리지 계정
→ 액세스 제어(IAM)
→ + 추가 → 역할 할당 추가
→ 역할: Storage Blob Data Contributor
→ 멤버: datacops-sp 선택
→ 검토 + 할당
```
 
### Step 2 — Databricks 앱에 Key Vault 권한 부여
 
```
Azure Portal
→ kv-sense-team4
→ 액세스 제어(IAM)
→ + 추가 → 역할 할당 추가
→ 역할: Key Vault 비밀 사용자
→ 멤버: "data" 검색 → Databricks (소문자) 선택
→ 검토 + 할당
```
 
> ⚠️ 권한 전파까지 최대 3분 소요. 할당 후 잠시 기다렸다가 노트북 실행.
 
### Step 3 — 권한 확인
 
Key Vault → 액세스 제어(IAM) → 역할 할당 탭에서 아래 두 항목 확인:
 
```
✅ datacops-sp   — Key Vault 비밀 사용자
✅ Databricks    — Key Vault 비밀 사용자
```
 
---
 
## 노트북 코드 전체
 
### 셀 1 — 패키지 설치
 
```python
# COMMAND ----------
%pip install azure-keyvault-secrets==4.7.0 azure-identity==1.15.0 azure-core==1.29.5 sseclient-py openai great-expectations
```
 
### 셀 2 — 재시작
 
```python
# COMMAND ----------
dbutils.library.restartPython()
```
 
### 셀 3 — Key Vault + ADLS + OpenAI 연결
 
```python
# COMMAND ----------
import os
import sys
import json
import requests
from sseclient import SSEClient
from datetime import datetime
 
# Key Vault 설정
os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")
 
import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager
 
vault = get_vault_manager()
print("[OK] Key Vault 연결 완료")
 
# ADLS 연결
storage_client = vault.get_storage_client("datacopsadls")
print("[OK] ADLS 연결 완료")
 
# OpenAI 키
openai_key      = vault.get_secret("azure-openai-key")
openai_endpoint = vault.get_secret("azure-openai-endpoint")
print("[OK] OpenAI 키 로드 완료")
```
 
### 셀 4 — Wikipedia SSE 수신
 
```python
# COMMAND ----------
WIKI_STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
headers = {
    "Accept": "text/event-stream",
    "User-Agent": "datacops-data-quality/0.1"
}
 
def collect_wiki_events(max_count=1000, wiki_filter=None):
    response = requests.get(WIKI_STREAM_URL, stream=True, headers=headers)
    client = SSEClient(response)
    events = []
 
    for event in client.events():
        if event.event != "message":
            continue
        try:
            data = json.loads(event.data)
        except json.JSONDecodeError:
            continue
 
        if wiki_filter and data.get("wiki") != wiki_filter:
            continue
 
        record = {
            "wiki":         data.get("wiki"),
            "type":         data.get("type"),
            "title":        data.get("title"),
            "user":         data.get("user"),
            "bot":          data.get("bot", False),
            "comment":      data.get("comment", ""),
            "length_new":   data.get("length", {}).get("new"),
            "length_old":   data.get("length", {}).get("old"),
            "timestamp":    data.get("timestamp"),
            "server_name":  data.get("server_name"),
            "collected_at": datetime.utcnow().isoformat()
        }
        events.append(record)
 
        if len(events) >= max_count:
            break
 
    return events
 
# 샘플 100건 수집
print("Wikipedia 이벤트 수집 시작...")
sample_events = collect_wiki_events(max_count=100)
print(f"[OK] {len(sample_events)}건 수집 완료")
 
import pandas as pd
df_sample = pd.DataFrame(sample_events)
display(df_sample.head(5))
```
 
### 셀 5 — Bronze 적재
 
```python
# COMMAND ----------
BRONZE_PATH = "abfss://pulse-bronze@datacopsadls.dfs.core.windows.net/wiki/"
 
df_spark = spark.createDataFrame(df_sample)
df_spark.write \
    .format("delta") \
    .mode("append") \
    .save(BRONZE_PATH)
 
print(f"[OK] Bronze 적재 완료: {df_spark.count()}건")
```
 
### 셀 6 — GPT-4o 규칙 생성 (최초 1회)
 
```python
# COMMAND ----------
from openai import AzureOpenAI
 
client_ai = AzureOpenAI(
    api_key=openai_key,
    azure_endpoint=openai_endpoint,
    api_version="2024-02-01"
)
 
columns_info = {
    col: {
        "dtype":     str(df_sample[col].dtype),
        "sample":    df_sample[col].dropna().head(5).tolist(),
        "null_rate": round(df_sample[col].isna().mean(), 3)
    }
    for col in df_sample.columns
}
 
prompt = f"""
당신은 데이터 품질 전문가입니다.
아래는 Wikipedia 실시간 편집 로그 데이터의 칼럼 정보입니다.
 
칼럼 정보:
{json.dumps(columns_info, ensure_ascii=False, indent=2)}
 
이 데이터에 적용할 데이터 품질 규칙을 생성해주세요.
반드시 아래 JSON 형식으로만 응답하세요:
 
{{
  "domain": "도메인명",
  "rules": [
    {{
      "column": "칼럼명",
      "expectation_type": "GX expectation 이름",
      "kwargs": {{}},
      "dimension": "completeness|validity|consistency|uniqueness|timeliness",
      "reason": "이 규칙이 필요한 이유"
    }}
  ]
}}
"""
 
response = client_ai.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "데이터 품질 규칙 생성 전문가입니다. JSON만 반환합니다."},
        {"role": "user", "content": prompt}
    ],
    response_format={"type": "json_object"}
)
 
rules_json = json.loads(response.choices[0].message.content)
print(f"[OK] 도메인 감지: {rules_json['domain']}")
print(f"[OK] 규칙 생성: {len(rules_json['rules'])}개")
for rule in rules_json["rules"]:
    print(f"  - {rule['column']}: {rule['expectation_type']} ({rule['dimension']})")
```
 
---
 
## 트러블슈팅
 
### `ModuleNotFoundError: No module named 'sseclient'`
 
```python
%pip install sseclient-py
dbutils.library.restartPython()
```
 
### `ImportError: cannot import name 'AccessTokenInfo'`
 
Azure SDK 버전 충돌. 버전 고정 설치:
 
```python
%pip install azure-keyvault-secrets==4.7.0 azure-identity==1.15.0 azure-core==1.29.5
dbutils.library.restartPython()
```
 
### `(Forbidden) Caller is not authorized`
 
Key Vault IAM에서 Databricks 앱 권한 미부여.  
→ Step 2 다시 확인. 권한 부여 후 3분 대기.
 
### `ValueError: Databricks Spark 설정에 필요한 시크릿이 누락`
 
Key Vault에 `adls-client-id`, `adls-client-secret`, `adls-tenant-id` 없음.  
→ Key Vault 비밀 목록 확인 후 추가.
 
---
 
## ADLS 컨테이너 경로
 
```
pulse-bronze     abfss://pulse-bronze@datacopsadls.dfs.core.windows.net/
pulse-silver     abfss://pulse-silver@datacopsadls.dfs.core.windows.net/
pulse-gold       abfss://pulse-gold@datacopsadls.dfs.core.windows.net/
pulse-quarantine abfss://pulse-quarantine@datacopsadls.dfs.core.windows.net/
```
 
---